# 01 — Limpeza de Dados
**Projeto Final — Data Analyst Junior · Bytes4Future**  
**Dataset:** Sales of Summer Clothes — Wish Platform (Kaggle, agosto 2020)  
**Responsável:** Gustavo  
**Referência:** `data_treatment_log.md` · `observacoes_externas.md`

---

## Objectivo deste notebook

Executar todas as decisões de limpeza documentadas no `data_treatment_log.md`.  
Cada célula corresponde a uma decisão — o comentário acima de cada bloco explica o **porquê**.

**Regra de ouro:** os dados originais nunca são alterados.  
Todas as correcções criam colunas novas (`ColNameFix`).  
O ficheiro `raw/` é intocável.

---

## Índice
1. Imports e carregamento
2. Verificação inicial
3. Remover colunas inúteis
4. Corrigir tipos de dados
5. Tratar duplicados — regra de negócio
6. RFIX_020 — rating 5.0 com rating_count 0
7. RFIX_005/021 — price > retail_price
8. has_urgency_banner_fix
9. product_color_fix
10. origin_country_fix
11. discount_pct_fix
12. units_sold_tier
13. distance_km
14. negative_rating_pct
15. total_badges
16. is_new_product_boost
17. Verificação final
18. Exportar processed

## 1. Imports e Carregamento

In [ ]:
import pandas as pd
import numpy as np

# Caminho do ficheiro raw — NUNCA alterar este ficheiro
RAW_PATH = '../../Datasets/Summer_Products.csv'
PROCESSED_PATH = '../../Datasets/Summer_Products_processado.csv'

# Carregar com encoding utf-8 (RFIX_019 — dados em francês com caracteres não-ASCII)
df = pd.read_csv(RAW_PATH, encoding='utf-8')

print(f'Dataset carregado: {df.shape[0]} linhas · {df.shape[1]} colunas')

Dataset carregado: 1573 linhas · 43 colunas


## 2. Verificação Inicial

In [2]:
# Shape e tipos
print('=== SHAPE ===')
print(df.shape)
print()
print('=== TIPOS ===')
print(df.dtypes)

=== SHAPE ===
(1573, 43)

=== TIPOS ===
title                            object
title_orig                       object
price                           float64
retail_price                      int64
currency_buyer                   object
units_sold                        int64
uses_ad_boosts                    int64
rating                          float64
rating_count                      int64
rating_five_count               float64
rating_four_count               float64
rating_three_count              float64
rating_two_count                float64
rating_one_count                float64
badges_count                      int64
badge_local_product               int64
badge_product_quality             int64
badge_fast_shipping               int64
tags                             object
product_color                    object
product_variation_size_id        object
product_variation_inventory       int64
shipping_option_name             object
shipping_option_price             int64


In [3]:
# Nulos por coluna
print('=== NULOS (%) ===')
nulls = (df.isnull().sum() / len(df) * 100).round(2)
print(nulls[nulls > 0].sort_values(ascending=False))

=== NULOS (%) ===
merchant_profile_picture     85.63
has_urgency_banner           69.93
urgency_text                 69.93
rating_five_count             2.86
rating_four_count             2.86
rating_three_count            2.86
rating_two_count              2.86
rating_one_count              2.86
product_color                 2.61
origin_country                1.08
product_variation_size_id     0.89
merchant_name                 0.25
merchant_info_subtitle        0.06
dtype: float64


In [4]:
# Estatísticas descritivas das colunas numéricas
print('=== DESCRIBE ===')
df.describe().round(2)

=== DESCRIBE ===


,price,retail_price,units_sold,uses_ad_boosts,rating,rating_count,rating_five_count,rating_four_count,rating_three_count,rating_two_count,...,badge_fast_shipping,product_variation_inventory,shipping_option_price,shipping_is_express,countries_shipped_to,inventory_total,has_urgency_banner,merchant_rating_count,merchant_rating,merchant_has_profile_picture
count,1573.00,1573.00,1573.00,1573.00,1573.00,1573.00,1528.00,1528.00,1528.00,1528.00,...,1573.00,1573.00,1573.00,1573.00,1573.00,1573.00,473.0,1573.00,1573.00,1573.00
mean,8.33,23.29,4339.01,0.43,3.82,889.66,442.26,179.60,134.55,63.71,...,0.01,33.08,2.35,0.00,40.46,49.82,1.0,26495.83,4.03,0.14
std,3.93,30.36,9356.54,0.50,0.52,1983.93,980.20,400.52,311.69,151.34,...,0.11,21.35,1.02,0.05,20.30,2.56,0.0,78474.46,0.20,0.35
min,1.00,1.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,...,0.00,1.00,1.00,0.00,6.00,1.00,1.0,0.00,2.33,0.00
25%,5.81,7.00,100.00,0.00,3.55,24.00,12.00,5.00,4.00,2.00,...,0.00,6.00,2.00,0.00,31.00,50.00,1.0,1987.00,3.92,0.00
50%,8.00,10.00,1000.00,0.00,3.85,150.00,79.00,31.50,24.00,11.00,...,0.00,50.00,2.00,0.00,40.00,50.00,1.0,7936.00,4.04,0.00
75%,11.00,26.00,5000.00,1.00,4.11,855.00,413.50,168.25,129.25,62.00,...,0.00,50.00,3.00,0.00,43.00,50.00,1.0,24564.00,4.16,0.00
max,49.00,252.00,100000.00,1.00,5.00,20744.00,11548.00,4152.00,3658.00,2003.00,...,1.00,50.00,12.00,1.00,140.00,50.00,1.0,2174765.00,5.00,1.00


## 3. Remover Colunas Inúteis

In [5]:
# Colunas a remover — documentadas no variables_definition.md
# Motivo: constantes (sem variação), URLs (não analíticas), texto não estruturado
cols_to_drop = [
    'currency_buyer',          # 100% EUR — sem variação
    'theme',                   # 100% 'summer' — sem variação
    'crawl_month',             # 100% '2020-08' — sem variação
    'title',                   # substituída por title_orig
    'product_url',             # URL — não analítico
    'product_picture',         # URL — não analítico
    'merchant_profile_picture',# 85.6% nulos — sem valor analítico
    'merchant_info_subtitle',  # texto bruto francês — RFIX_015 e RFIX_016
    'inventory_total',         # 99.4% no cap de 50 — RFIX_017
]

df = df.drop(columns=cols_to_drop)
print(f'Colunas após remoção: {df.shape[1]}')
print('Colunas restantes:', list(df.columns))

Colunas após remoção: 34
Colunas restantes: ['title_orig', 'price', 'retail_price', 'units_sold', 'uses_ad_boosts', 'rating', 'rating_count', 'rating_five_count', 'rating_four_count', 'rating_three_count', 'rating_two_count', 'rating_one_count', 'badges_count', 'badge_local_product', 'badge_product_quality', 'badge_fast_shipping', 'tags', 'product_color', 'product_variation_size_id', 'product_variation_inventory', 'shipping_option_name', 'shipping_option_price', 'shipping_is_express', 'countries_shipped_to', 'has_urgency_banner', 'urgency_text', 'origin_country', 'merchant_title', 'merchant_name', 'merchant_rating_count', 'merchant_rating', 'merchant_id', 'merchant_has_profile_picture', 'product_id']


## 4. Corrigir Tipos de Dados

In [6]:
# Forçar tipos correctos para evitar erros nos cálculos
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['retail_price'] = pd.to_numeric(df['retail_price'], errors='coerce')
df['units_sold'] = pd.to_numeric(df['units_sold'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating_count'] = pd.to_numeric(df['rating_count'], errors='coerce')
df['shipping_option_price'] = pd.to_numeric(df['shipping_option_price'], errors='coerce')
df['countries_shipped_to'] = pd.to_numeric(df['countries_shipped_to'], errors='coerce')
df['merchant_rating'] = pd.to_numeric(df['merchant_rating'], errors='coerce')
df['merchant_rating_count'] = pd.to_numeric(df['merchant_rating_count'], errors='coerce')
df['product_id'] = df['product_id'].astype(str)
df['merchant_id'] = df['merchant_id'].astype(str)

print('Tipos corrigidos.')
print(df[['price','retail_price','units_sold','rating','rating_count']].dtypes)

Tipos corrigidos.
price           float64
retail_price      int64
units_sold        int64
rating          float64
rating_count      int64
dtype: object


## 5. Tratar Duplicados — Regra de Negócio

In [7]:
# RFIX_007 + RFIX_009 + RFIX_010
# Existem 232 duplicados por product_id
# Causa: scraping em múltiplos momentos + testes A/B do Wish (urgency banner on/off)

print(f'Linhas antes: {len(df)}')
print(f'product_id únicos: {df["product_id"].nunique()}')
print(f'Duplicados: {len(df) - df["product_id"].nunique()}')

# Verificar quantos duplicados têm urgency_text numa versão e não na outra
dup_ids = df[df.duplicated(subset=['product_id'], keep=False)]['product_id'].unique()
print(f'\nproduct_id com duplicados: {len(dup_ids)}')

Linhas antes: 1573
product_id únicos: 1341
Duplicados: 232

product_id com duplicados: 211


In [8]:
# REGRA DE NEGÓCIO — documentada em data_treatment_log.md
# Objectivo da análise: estudar impacto de urgency banners no consumo
# Decisão: para cada product_id duplicado, manter a linha COM urgency_text
# Motivo: queremos o estado com maior pressão de compra (análise ESG de impulso)
# Alternativa rejeitada: manter a primeira ocorrência — perderia informação de banner

# Ordenar: linhas com urgency_text não-nulo ficam primeiro
df['_has_urgency_sort'] = df['urgency_text'].notna().astype(int)
df = df.sort_values('_has_urgency_sort', ascending=False)

# Manter primeira ocorrência (que agora é a que tem urgency_text quando existe)
df = df.drop_duplicates(subset=['product_id'], keep='first').copy()
df = df.drop(columns=['_has_urgency_sort'])

print(f'Linhas após drop_duplicates: {len(df)}')
print(f'Esperado: ~1341 produtos únicos')

Linhas após drop_duplicates: 1341
Esperado: ~1341 produtos únicos


## 6. RFIX_020 — Rating 5.0 com rating_count = 0

In [9]:
# RFIX_020 — confirmado pelo criador do dataset (Jeffrey Mvutu Mabilama)
# O Wish atribui automaticamente 5 estrelas a produtos novos sem avaliações reais
# É um boost artificial do algoritmo de recomendação — não reflecte qualidade real
#
# Decisão: criar flag binária is_new_product_boost
# E substituir rating por NaN nestes casos para não distorcer médias de satisfação
#
# Alternativa rejeitada: manter rating 5.0 — distorceria a análise ESG de qualidade

mask_boost = (df['rating'] == 5.0) & (df['rating_count'] == 0)

df['is_new_product_boost'] = mask_boost.astype(int)
df['rating_fix'] = df['rating'].copy()
df.loc[mask_boost, 'rating_fix'] = np.nan

print(f'Produtos com boost artificial: {mask_boost.sum()}')
print(f'rating_fix nulos: {df["rating_fix"].isna().sum()}')
print('\nVerificação — is_new_product_boost:')
print(df['is_new_product_boost'].value_counts())

Produtos com boost artificial: 35
rating_fix nulos: 35

Verificação — is_new_product_boost:
is_new_product_boost
0    1306
1      35
Name: count, dtype: int64


## 7. RFIX_005 / RFIX_021 — price > retail_price

In [10]:
# RFIX_005 + RFIX_021
# Em 559 casos (35%), price > retail_price
# Causa confirmada pelo criador: bug de moedas misturadas durante o scraping
# (a página alternava entre USD e EUR abruptamente)
# NÃO é estratégia de preço, NÃO é efeito COVID — é erro técnico de captura
#
# Decisão: criar coluna booleana price_above_retail para identificar estes casos
# Estes registos serão excluídos de análises de desconto mas mantidos no dataset
#
# Alternativa rejeitada: excluir as 559 linhas — perderíamos 35% do dataset

df['price_above_retail'] = (df['price'] > df['retail_price']).astype(int)

print(f'price > retail_price: {df["price_above_retail"].sum()} casos')
print(f'% do dataset: {df["price_above_retail"].mean()*100:.1f}%')

price > retail_price: 477 casos
% do dataset: 35.6%


## 8. has_urgency_banner_fix

In [11]:
# NaN em has_urgency_banner = ausência de banner (não é dado em falta)
# A plataforma só preenche o campo quando há banner — RFIX_008
# Decisão: NaN → 0
# Alternativa rejeitada: excluir os 1100 registos — eliminaria 70% do dataset

df['has_urgency_banner_fix'] = df['has_urgency_banner'].fillna(0).astype(int)

print('has_urgency_banner_fix:')
print(df['has_urgency_banner_fix'].value_counts())
print(f'Nulos restantes: {df["has_urgency_banner_fix"].isna().sum()}')

has_urgency_banner_fix:
has_urgency_banner_fix
0    873
1    468
Name: count, dtype: int64
Nulos restantes: 0


## 9. product_color_fix

In [12]:
# 41 nulos em product_color
# Decisão: NaN → 'unknown'
# Alternativa rejeitada: imputar cor mais frequente ('black') — seria inventar dados

df['product_color_fix'] = df['product_color'].fillna('unknown')

print(f'Nulos antes: {df["product_color"].isna().sum()}')
print(f'Nulos depois: {df["product_color_fix"].isna().sum()}')
print(f'Valores únicos: {df["product_color_fix"].nunique()}')

Nulos antes: 40
Nulos depois: 0
Valores únicos: 102


## 10. origin_country_fix

In [13]:
# 17 nulos em origin_country
# Decisão: NaN → 'unknown'
# Alternativa rejeitada: imputar 'CN' (moda, 96%) — seria especular sobre origem
# Estes 17 registos serão excluídos do cálculo de distance_km

df['origin_country_fix'] = df['origin_country'].fillna('unknown')

print('origin_country_fix:')
print(df['origin_country_fix'].value_counts(dropna=False))

origin_country_fix:
origin_country_fix
CN         1294
US           27
unknown      13
VE            3
SG            2
GB            1
AT            1
Name: count, dtype: int64


## 11. discount_pct_fix

In [14]:
# Calcular desconto real — sem valores negativos
# Casos onde price > retail_price (bug de moedas — RFIX_021) → desconto = 0
# Fórmula: (retail_price - price) / retail_price * 100
# Negativos → 0 (sem desconto real)

def calc_discount(row):
    if row['retail_price'] == 0 or pd.isna(row['retail_price']):
        return np.nan
    discount = (row['retail_price'] - row['price']) / row['retail_price'] * 100
    return max(round(discount, 2), 0)

df['discount_pct_fix'] = df.apply(calc_discount, axis=1)

print('discount_pct_fix:')
print(df['discount_pct_fix'].describe().round(2))
print(f'Valores negativos: {(df["discount_pct_fix"] < 0).sum()} (deve ser 0)')

discount_pct_fix:
count    1341.00
mean       31.50
std        36.55
min         0.00
25%         0.00
50%         7.00
75%        72.73
max        96.93
Name: discount_pct_fix, dtype: float64
Valores negativos: 0 (deve ser 0)


## 12. units_sold_tier

In [15]:
# units_sold são buckets do Wish — não valores contínuos reais (RFIX_006 + RFIX_022)
# Tratar como variável ordinal em 5 níveis
# Alternativa rejeitada: manter contínuo — correlações seriam matematicamente
# correctas mas analiticamente enganosas (buckets vs valores reais)

def units_to_tier(val):
    if pd.isna(val):   return np.nan
    if val <= 10:      return 1  # Volume muito baixo
    if val <= 100:     return 2  # Volume baixo
    if val <= 1000:    return 3  # Volume médio
    if val <= 10000:   return 4  # Volume alto
    return 5                     # Volume muito alto

df['units_sold_tier'] = df['units_sold'].apply(units_to_tier).astype('Int64')

print('units_sold_tier:')
print(df['units_sold_tier'].value_counts().sort_index())
print(f'Nulos: {df["units_sold_tier"].isna().sum()}')

units_sold_tier:
units_sold_tier
1     48
2    446
3    362
4    363
5    122
Name: count, dtype: Int64
Nulos: 0


## 13. distance_km

In [16]:
# Distância estimada do país de origem até Paris (Europa Ocidental)
# Metodologia A — distância fixa por país (ver data_treatment_log.md)
# Alternativa B rejeitada: countries_shipped_to é contagem, não lista de países
# Alternativa C rejeitada: API de geocoding — desnecessária com 6 países de origem

distance_map = {
    'CN': 9200,   # China → Paris
    'US': 8500,   # USA → Paris
    'VE': 8300,   # Venezuela → Paris
    'SG': 10200,  # Singapura → Paris
    'AT': 1050,   # Áustria → Paris
    'GB': 340,    # Reino Unido → Paris
}

df['distance_km'] = df['origin_country_fix'].map(distance_map)
# 'unknown' ficará como NaN — excluído de análises de frete

print('distance_km por país de origem:')
print(df.groupby('origin_country_fix')['distance_km'].first())
print(f'\nNulos (unknown): {df["distance_km"].isna().sum()}')

distance_km por país de origem:
origin_country_fix
AT          1050.0
CN          9200.0
GB           340.0
SG         10200.0
US          8500.0
VE          8300.0
unknown        NaN
Name: distance_km, dtype: float64

Nulos (unknown): 13


## 14. negative_rating_pct

In [17]:
# % de avaliações negativas (1 estrela + 2 estrelas) sobre o total
# O rating médio esconde a distribuição — este campo revela a insatisfação real
# 45 registos sem breakdown de rating → NaN (não excluídos do dataset)

def calc_neg_rating(row):
    if pd.isna(row['rating_count']) or row['rating_count'] == 0:
        return np.nan
    if pd.isna(row['rating_one_count']) or pd.isna(row['rating_two_count']):
        return np.nan
    neg = row['rating_one_count'] + row['rating_two_count']
    return round(neg / row['rating_count'] * 100, 2)

df['negative_rating_pct'] = df.apply(calc_neg_rating, axis=1)

print('negative_rating_pct:')
print(df['negative_rating_pct'].describe().round(2))
print(f'Nulos: {df["negative_rating_pct"].isna().sum()}')

negative_rating_pct:
count    1306.00
mean       19.88
std        11.93
min         0.00
25%        12.33
50%        17.92
75%        25.30
max       100.00
Name: negative_rating_pct, dtype: float64
Nulos: 35


## 15. total_badges

In [18]:
# Soma dos 3 badges — confirmado pelo criador como soma simples (RFIX_013)
# ATENÇÃO: não usar total_badges E as colunas individuais simultaneamente
# em modelos — multicolinearidade perfeita (RFIX_013)

df['total_badges'] = (
    df['badge_local_product'] +
    df['badge_product_quality'] +
    df['badge_fast_shipping']
).astype(int)

print('total_badges:')
print(df['total_badges'].value_counts().sort_index())

total_badges:
total_badges
0    1204
1     125
2      10
3       2
Name: count, dtype: int64


## 16. Verificação Final

In [19]:
print('=== CHECKLIST FINAL ===')
print()

checks = {
    'Linhas totais (~1341)': len(df),
    'has_urgency_banner_fix nulos': df['has_urgency_banner_fix'].isna().sum(),
    'product_color_fix nulos': df['product_color_fix'].isna().sum(),
    'origin_country_fix nulos': df['origin_country_fix'].isna().sum(),
    'discount_pct_fix negativos': (df['discount_pct_fix'] < 0).sum(),
    'units_sold_tier fora de 1-5': (~df['units_sold_tier'].isin([1,2,3,4,5])).sum(),
    'is_new_product_boost total': df['is_new_product_boost'].sum(),
    'rating_fix nulos (boost)': df['rating_fix'].isna().sum(),
    'price_above_retail total': df['price_above_retail'].sum(),
    'total_badges fora de 0-3': (~df['total_badges'].isin([0,1,2,3])).sum(),
}

for k, v in checks.items():
    status = '✅' if v == 0 or 'total' in k.lower() or 'linhas' in k.lower() else '❌'
    print(f'{status}  {k}: {v}')

=== CHECKLIST FINAL ===

✅  Linhas totais (~1341): 1341
✅  has_urgency_banner_fix nulos: 0
✅  product_color_fix nulos: 0
✅  origin_country_fix nulos: 0
✅  discount_pct_fix negativos: 0
✅  units_sold_tier fora de 1-5: 0
✅  is_new_product_boost total: 35
❌  rating_fix nulos (boost): 35
✅  price_above_retail total: 477
✅  total_badges fora de 0-3: 0


In [20]:
# Resumo das colunas no dataset final
print('=== COLUNAS FINAIS ===')
print(f'Total: {df.shape[1]} colunas')
print()
originais = [c for c in df.columns if not c.endswith('_fix') and not c.endswith('_tier') 
             and not c.endswith('_pct') and not c.endswith('_km') 
             and not c.endswith('_boost') and c != 'total_badges']
novas = [c for c in df.columns if c not in originais]
print(f'Originais mantidas: {len(originais)}')
print(f'Novas (Fix/calculadas): {len(novas)}')
print('\nColunas novas:', novas)

=== COLUNAS FINAIS ===
Total: 45 colunas

Originais mantidas: 35
Novas (Fix/calculadas): 10

Colunas novas: ['is_new_product_boost', 'rating_fix', 'has_urgency_banner_fix', 'product_color_fix', 'origin_country_fix', 'discount_pct_fix', 'units_sold_tier', 'distance_km', 'negative_rating_pct', 'total_badges']


## 17. Exportar Processed

In [21]:
# Guardar o dataset limpo em 01_data/processed/
# Este é o ficheiro que todos os outros notebooks vão usar

df.to_csv(PROCESSED_PATH, index=False, encoding='utf-8')

# Verificação
df_check = pd.read_csv(PROCESSED_PATH)
print(f'✅ Ficheiro guardado: {PROCESSED_PATH}')
print(f'   Linhas: {df_check.shape[0]}')
print(f'   Colunas: {df_check.shape[1]}')
print()
print('Primeiras colunas novas no ficheiro exportado:')
new_cols = ['has_urgency_banner_fix', 'product_color_fix', 'origin_country_fix',
            'discount_pct_fix', 'units_sold_tier', 'distance_km',
            'negative_rating_pct', 'total_badges', 'is_new_product_boost',
            'rating_fix', 'price_above_retail']
print(df_check[new_cols].head(3))

✅ Ficheiro guardado: ../../Datasets/Summer_Products_processado.csv
   Linhas: 1341
   Colunas: 45

Primeiras colunas novas no ficheiro exportado:
   has_urgency_banner_fix product_color_fix origin_country_fix  \
0                       1             white                 CN   
1                       1             white                 CN   
2                       1          navyblue                 CN   

   discount_pct_fix  units_sold_tier  distance_km  negative_rating_pct  \
0              0.00                2       9200.0                18.52   
1             71.05                4       9200.0                16.75   
2              0.00                5       9200.0                12.93   

   total_badges  is_new_product_boost  rating_fix  price_above_retail  
0             0                     0        3.76                   1  
1             0                     0        3.91                   0  
2             0                     0        4.06                   1  
